In [ ]:

%load_ext autoreload

%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from duckdb import sql as sqldf

from pathlib import Path
from datetime import datetime


import matplotlib.pyplot as plt


In [ ]:
from Rockets.Superformula.Formulas import formula1
from Rockets.Geometry import grain
from Rockets.Simulation.Plots import Web

In [ ]:
np.set_printoptions(floatmode='fixed', precision=6,  suppress=True)

In [ ]:
pwd

In [ ]:
import numpy as np
from Rockets.Kang.saburnbackfinal import solve_burnback, pressure_profile

A_MIN = 0.1
A_MAX = 0.8

B_MIN = 0.1
B_MAX = 0.8

M_MIN = 1
M_MAX = 10

N1_MIN = 0.1
N1_MAX = 15.0

N2_MIN = 0.1
N2_MAX = 15.0

N3_MIN = 0.1
N3_MAX = 15.0

PC0 = 101325.0
MPA = 1e6

In [ ]:
m_ref  = np.random.randint(M_MIN, M_MAX)
a_ref  = np.random.uniform(A_MIN, A_MAX)
b_ref  = a_ref
n1_ref = np.random.uniform(N1_MIN, N1_MAX)
n2_ref = np.random.uniform(N2_MIN, N2_MAX)
n3_ref = n2_ref

In [ ]:
inpt = dict(m=m_ref, a=a_ref ,b = b_ref ,n_1 = n1_ref, n_2 = n2_ref, n_3= n3_ref)
inpt

In [ ]:

inpt = {'m': 9,
 'a': 0.5,
 'b': 0.5,
 'n_1': 5,
 'n_2': 10,
 'n_3': 10}

In [ ]:
m_ref ,a_ref ,b_ref ,n1_ref,n2_ref,n3_ref = inpt.values()

In [ ]:
SFparams = {
'M': m_sf,
'A': a_sf,
'B': b_sf,
'N1': n1_sf,
'N2': n2_sf,
'N3': n3_sf,
}

simparams = {
'NMO': nm_o,
'NMI': nm_i,
'CPF': C_pf,
'RO': Ro,
'RI': Ri
}
"""

## Preview initial shape

In [ ]:
f1 = formula1(o=0, **inpt)
f1

In [ ]:
I, RT, XY  = grain(f1, 1000)


In [ ]:
px.scatter(x = XY[:,0], y = XY[:,1], width=500)

## Run the FEM simulation

In [ ]:
mesh_ref, S_ref = solve_burnback(a_ref, b_ref, m_ref, 
                                 n1_ref, n2_ref, n3_ref, 
                                 nm_o=500, nm_i=250, plot=True)


In [ ]:
x, y = mesh_ref.x, mesh_ref.y

In [ ]:
z = (S_ref*100).astype(int)

## View simulation results

In [ ]:
px.scatter(x = x, y = y,color = z, width=500)

In [ ]:
px.line(x = x, y = y,color = z, width=800, height=700)

In [ ]:
px.line_3d(x = x, y = y,color = z, z = S_ref, width=800, height=700)

In [ ]:
plt.ioff()

In [ ]:
import matplotlib.tri as mtri

triangles = mesh_ref.triangles[:, :3] - 1  # convert to zero-based indices
tri = mtri.Triangulation(mesh_ref.x, mesh_ref.y, triangles=triangles)

levels = [k*1e-2 for k in range(1,101)]


In [ ]:

contours = plt.tricontour(tri, S_ref, levels=levels,)
paths = contours.allsegs[0]  # separate contour paths; each is an array of (x, y)

In [ ]:
px.line(S_ref, render_mode='svg')

In [ ]:
px.histogram(S_ref)

## Transform raw mesh simulation data into curves of shape evolution

In [ ]:
DT = 0.01
S = S_ref
n_levels = int(np.max(S)/DT) + 1
web_vals = np.linspace(0, np.max(S), n_levels)

In [ ]:
web_vals

In [ ]:
wat = contours.allsegs[0][0]
wat.shape

In [ ]:
def emit(From, web_vals):
    for i,lvl in enumerate(web_vals): 
        for j,p in enumerate(From.allsegs[i]):
            p = np.vstack([p,[np.nan, np.nan]])
            idx = np.arange(p.shape[0])
            yield {"I": idx, "X": p[:,0], "Y": p[:,1], 
                   "SimStep": i, "lvl":lvl ,"patch": j }



In [ ]:
import matplotlib.tri as mtri
plt.ioff()
triangles = mesh_ref.triangles[:, :3] - 1  # convert to zero-based indices
tri = mtri.Triangulation(mesh_ref.x, mesh_ref.y, triangles=triangles)

web_vals = np.linspace(0, np.max(S), n_levels)
contours = plt.tricontour(tri, S_ref, levels=web_vals,)

paths = contours.allsegs[0]  # separate contour paths; each is an array of (x, y)

DT = 0.01
S = S_ref
n_levels = int(np.max(S)/DT) + 1

In [ ]:
curves = [pd.DataFrame(data) for data in emit(contours, web_vals)]
curves = pd.concat(curves)
curves["StepMod10"] = curves.SimStep.astype(int) % 10
curves = curves.sort_values(["SimStep","patch","I"])
curves

In [ ]:
npoints = curves.groupby("SimStep", as_index=False).count()
npoints

In [ ]:
px.line(npoints, x = "SimStep", y = "X")

In [ ]:
fig = px.line(curves, x='X', y='Y', color='StepMod10', render_mode='SVG', width=800)

In [ ]:
fig.add_trace(go.Scatter(x = XY[:,0], y = XY[:,1], name="initial"))
fig.update_yaxes(scaleanchor="x", scaleratio=1)

In [ ]:
ts_ref, Ps_ref, _ = pressure_profile( mesh_ref, S_ref, plot=False)


In [ ]:
px.line(x = ts_ref, y = Ps_ref,)